In [ ]:
import pandas as pd
import os
import json
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)
from importlib import reload


In [ ]:
###### LOADING THE TWO DATASETS I NEED

In [ ]:
fpath = curr_wd + "/data/input/RG_motif_info_df.parquet"
motif_info_set_df = pd.read_parquet(fpath)
motif_info_set_df

In [ ]:
fpath = curr_wd + "/data/input/all_IDR_human.parquet"
IDR_info_df = pd.read_parquet(fpath)
IDR_info_df.rename(columns={"protein_name": "UniqueID"}, inplace=True)
IDR_info_df

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

def merge_motif_idr(df_motif, df_idr):
    df = df_motif.merge(
        df_idr,
        on="UniqueID",
        how="left",
        validate="many_to_one"
    )
    return df

def extract_window(row, flank=20):
    seq = row["full_seq"]
    prot_len = len(seq)

    motif_start = row["start"]
    motif_end = row["end"]

    win_start = max(0, motif_start - flank)
    win_end = min(prot_len, motif_end + flank)

    window_seq = seq[win_start:win_end]

    return window_seq, win_start, win_end


def idr_fraction_interval(idr_array, start, end):
    try:
        window = idr_array[start:end]
    except Exception:
        return None

    if len(window) == 0:
        return None

    return sum(window) / len(window)

def effective_flank_limits(row, flank):
    motif_start = row["start"]
    motif_end = row["end"]
    prot_len = len(row["full_seq"])

    max_left = motif_start
    max_right = prot_len - motif_end

    return min(flank, max_left), min(flank, max_right)
def adaptive_flank_trimming(row,
                             flank=20,
                             min_idr_fraction=0.7,
                             min_flank_len=5,
                             trim_step=1):

    seq = row["full_seq"]
    idr = row["prediction-disorder-mobidb_lite"]
    prot_len = len(seq)

    motif_start = row["start"]
    motif_end = row["end"]

    left_flank, right_flank = effective_flank_limits(row, flank)

    # --- Trim left flank ---
    while left_flank >= min_flank_len:
        l_start = motif_start - left_flank
        l_end = motif_start

        frac = idr_fraction_interval(idr, l_start, l_end)
        if frac is not None and frac >= min_idr_fraction:
            break

        left_flank -= trim_step

    # --- Trim right flank ---
    while right_flank >= min_flank_len:
        r_start = motif_end
        r_end = motif_end + right_flank

        frac = idr_fraction_interval(idr, r_start, r_end)
        if frac is not None and frac >= min_idr_fraction:
            break

        right_flank -= trim_step

    return left_flank, right_flank


def extract_adaptive_window(row,
                            flank=20,
                            min_idr_fraction=0.7,
                            min_flank_len=5,
                            trim_step=1):

    left_flank, right_flank = adaptive_flank_trimming(
        row,
        flank=flank,
        min_idr_fraction=min_idr_fraction,
        min_flank_len=min_flank_len,
        trim_step=trim_step
    )

    seq = row["full_seq"]
    motif_start = row["start"]
    motif_end = row["end"]

    win_start = max(0, motif_start - left_flank)
    win_end = min(len(seq), motif_end + right_flank)

    window_seq = seq[win_start:win_end]

    return window_seq, win_start, win_end, left_flank, right_flank

def extract_control_window(row, flank=20):
    seq = row["full_seq"]
    prot_len = len(seq)

    motif_start = row["start"]
    motif_end = row["end"]

    left_flank = min(flank, motif_start)
    right_flank = min(flank, prot_len - motif_end)

    win_start = motif_start - left_flank
    win_end = motif_end + right_flank

    window_seq = seq[win_start:win_end]

    return window_seq, win_start, win_end, left_flank, right_flank



def build_fasta_entries(df,
                        flank=20,
                        mode="adaptive",  # "adaptive" or "control"
                        min_idr_fraction=0.7,
                        min_flank_len=5,
                        trim_step=1):

    fasta_records = []
    meta_records = []

    for _, row in df.iterrows():

        if mode == "adaptive":
            window_seq, win_start, win_end, lf, rf = extract_adaptive_window(
                row,
                flank=flank,
                min_idr_fraction=min_idr_fraction,
                min_flank_len=min_flank_len,
                trim_step=trim_step
            )
        elif mode == "control":
            window_seq, win_start, win_end, lf, rf = extract_control_window(
                row,
                flank=flank
            )
        elif mode == "full":
            seq = row["full_seq"]
            # win_start, win_end = 0, len(seq)
            # window_seq = seq
            # Compute IDR-aware flanks for metadata, but don't crop the sequence
            window_seq, win_start, win_end, lf, rf = extract_adaptive_window(
                row,
                flank=flank,
                min_idr_fraction=min_idr_fraction,
                min_flank_len=min_flank_len,
                trim_step=trim_step
            )
            window_seq = seq
        else:
            raise ValueError(f"Unknown mode: {mode}")

        if len(window_seq) == 0:
            continue

        header = (
            f">{row['UniqueID']}_{win_start}-{win_end}"
            f"|prot_range={win_start}-{win_end}"
            f"|motif_range={row['start']}-{row['end']}"
            f"|Lflank={lf}"
            f"|Rflank={rf}"
            f"|mode={mode}"
        )

        fasta_records.append((header, window_seq))

        meta_records.append({
            "UniqueID": row["UniqueID"],
            "orig_motif_index": row["orig_motif_index"],
            # "group": row["group"],            # pos / neg
            "mode": mode,                     # adaptive / control
            "motif_start": row["start"],
            "motif_end": row["end"],
            "motif_length": row["end"] - row["start"],
            "win_start": win_start,
            "win_end": win_end,
            "window_length": win_end - win_start,
            "left_flank": lf,
            "right_flank": rf
        })

    return fasta_records, pd.DataFrame(meta_records)


def write_fasta(records, out_fasta):
    with open(out_fasta, "w") as f:
        for header, seq in records:
            f.write(header + "\n")
            f.write(seq + "\n")

def write_fasta_chunked(records, out_fasta, max_residues=100_000):
    out_path = Path(out_fasta)
    stem = out_path.stem
    suffix = out_path.suffix
    parent = out_path.parent

    chunk_idx = 1
    current_records = []
    current_residues = 0
    written_files = []

    for header, seq in records:
        seq_len = len(seq)

        # If adding this sequence would exceed the limit, flush current chunk first
        if current_residues + seq_len > max_residues and current_records:
            out_chunk = parent / f"{stem}_part{chunk_idx}{suffix}"
            write_fasta(current_records, out_chunk)
            print(f"Wrote {len(current_records)} sequences ({current_residues} residues) to {out_chunk}")
            written_files.append(str(out_chunk))
            chunk_idx += 1
            current_records = []
            current_residues = 0

        current_records.append((header, seq))
        current_residues += seq_len

    # Flush final chunk
    if current_records:
        out_chunk = parent / f"{stem}_part{chunk_idx}{suffix}"
        write_fasta(current_records, out_chunk)
        print(f"Wrote {len(current_records)} sequences ({current_residues} residues) to {out_chunk}")
        written_files.append(str(out_chunk))

    return written_files

def motifs_to_blast_fasta(df_motif,
                          df_idr,
                          out_fasta,
                          flank=10,
                          mode="adaptive",
                          min_idr_fraction=0.7,
                          min_flank_len=5,
                          trim_step=1):

    # if mode == "full":
    #     df = df_motif
    # else:
    df = merge_motif_idr(df_motif, df_idr)

    # --- Compute full motif-level metadata (all motifs, all flanks) ---
    _, df_windows = build_fasta_entries(
        df,
        flank=flank,
        mode=mode,
        min_idr_fraction=min_idr_fraction,
        min_flank_len=min_flank_len,
        trim_step=trim_step
    )

    # --- Build deduplicated FASTA (one entry per unique protein) ---
    if mode == "full":
        # For full mode, deduplicate on UniqueID — take first occurrence
        df_dedup = df.drop_duplicates(subset="UniqueID")
    else:
        # For windowed modes, the window is motif-specific so no deduplication makes sense
        # Deduplicate on (UniqueID, win_start, win_end) via the records themselves
        df_dedup = df.drop_duplicates(subset="UniqueID")

    fasta_records, _ = build_fasta_entries(
        df_dedup,
        flank=flank,
        mode=mode,
        min_idr_fraction=min_idr_fraction,
        min_flank_len=min_flank_len,
        trim_step=trim_step
    )

    # written_files = write_fasta_chunked(fasta_records, out_fasta, max_residues=100_000)
    written_files = None
    print(f"Wrote {len(fasta_records)} deduplicated sequences ({mode})")
    print(f"Full motif metadata: {len(df_windows)} motif entries across {df_windows['UniqueID'].nunique()} proteins")

    return df_windows, written_files

df_windows_adaptive_neg, written_files = motifs_to_blast_fasta(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "7"],
    df_idr=IDR_info_df,
    out_fasta="/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/blastp/input/RG_motifs_flank5_IDR90_adaptive_neg_test.fasta",
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

df_windows_adaptive_pos, written_files = motifs_to_blast_fasta(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "5"],
    df_idr=IDR_info_df,
    out_fasta="/mnt/d/phd/scripts/15_vertical_evolutionary_analysis_orthologs/data/blastp/input/RG_motifs_flank5_IDR90_adaptive_pos_test.fasta",
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

In [ ]:
df_windows_adaptive_neg

In [ ]:
# from window_extraction import prepare_window_dataframe
import src.window_extraction as window_extraction
# Reload the module after making changes
reload(window_extraction)

# from fasta_writer import build_fasta_records, write_fasta_chunked

df_windows_neg = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "7"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)
print(df_windows_neg.head())
print(df_windows_neg.shape)

df_windows_pos = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "5"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_pos.head())
print(df_windows_pos.shape)

In [ ]:
###### SAVE THE METADATA FOR LATER

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "pos_RG_regions_win5_metadata.pkl"

df_windows_pos.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "neg_RG_regions_win5_metadata.pkl"

df_windows_neg.to_pickle(os.path.join(path, filename))


In [ ]:
#### GENERATE JSON FILES FOR GNOMAD

result_neg = df_windows_neg[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "neg_RG_regions_win5.json"


with open(f"{path}/{filename}", "w") as f:
    json.dump(result_neg, f, indent=2)

#######################################

result_pos = df_windows_pos[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "pos_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

In [ ]:
##### GENERATE FASTA FILES FOR BLASTP

import src.fasta_writer as fasta_writer
reload(fasta_writer)
# --- STEP 2: write FASTA ---
records_neg = fasta_writer.build_fasta_records(df_windows_neg)

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"
filename = "neg_RG_regions_win5.fasta"

written_files = fasta_writer.write_fasta_chunked(
    records_neg,
    f"{path}/{filename}"
)

# ________________________

records_pos = fasta_writer.build_fasta_records(df_windows_pos)

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"
filename = "pos_RG_regions_win5.fasta"

written_files = fasta_writer.write_fasta_chunked(
    records_pos,
    f"{path}/{filename}"
)